# WASM Compatibility Analysis for Positorium

Based on the `cargo check --target wasm32-unknown-unknown` output, here is the breakdown of the blockers:

## 1. Non-WASM Compatible Dependencies
The following dependencies or their features are incompatible with the `wasm32-unknown-unknown` target:

- **`tokio` / `mio`**: The current build uses `rt-multi-thread` which pulls in `mio`. `mio` does not support WASM targets. 
    - *Action:* Move networking (`axum`, `tokio`, `tower-http`) behind a `server` feature flag. Web builds will use the engine directly as a library.
- **`rusqlite`**: Relies on C-based SQLite, which is difficult (though not impossible) to compile to WASM.
    - *Action:* Move persistence (`rusqlite` and `src/persist.rs`) behind a `persistence` feature flag.
- **`config`**: Typically reads from the file system, which is limited in WASM.
    - *Action:* Move CLI/Config loading behind a `cli` feature flag.
- **`chrono`**: Requires `js-sys` or similar for WASM time support if using system time.

## 2. Proposed Feature Flags Architecture
To support both a full-featured server and a lightweight WASM library:

| Feature | Description | Default | Dependencies |
| :--- | :--- | :--- | :--- |
| `server` | Axum HTTP server and REST API | Yes | `axum`, `tokio`, `tower-http` |
| `persistence` | SQLite file-backed storage | Yes | `rusqlite` |
| `cli` | Command-line interface and file config | Yes | `config` |

## 3. Next Steps
1. Refactor `Cargo.toml` to define these features.
2. Use `#[cfg(feature = "...")]` in `src/lib.rs` and `src/main.rs`.
3. Extract the core engine logic in `src/traqula.rs` and `src/construct.rs` to be feature-agnostic.